### 1. 数学定义

假设有两个概率分布 $P$ 和 $Q$，其中 $P$ 代表**真实分布**，$Q$ 代表**模型预测的分布**。

KL 散度的定义为：

$$
D_{KL}(P || Q) = \sum_{x} P(x) \log \left( \frac{P(x)}{Q(x)} \right) = \sum_{x} P(x) (\log P(x) - \log Q(x))
$$

**关键特性：**
- **非对称性**：$D_{KL}(P || Q) \neq D_{KL}(Q || P)$，所以它不是严格意义上的"距离"
- **非负性**：$D_{KL}(P || Q) \geq 0$，当且仅当 $P = Q$ 时等于 0


### 2. PyTorch 实现与常见陷阱

PyTorch 提供了 `torch.nn.functional.kl_div`，但使用时需要注意两个关键点：

#### ⚠️ 陷阱 1：输入格式不对称
- **第一个参数 (input)**：必须是 **Log Probabilities** (`log_softmax` 的输出)
- **第二个参数 (target)**：是普通的 **Probabilities**

这样设计是为了**数值稳定性**。通过要求用户传入 `log_softmax` 的结果，PyTorch 利用了 Log-Sum-Exp 技巧，避免了 $\log(0)$ 和梯度爆炸问题。

#### ⚠️ 陷阱 2：`reduction='batchmean'` 的行为
- `reduction='sum'`：直接求和所有元素
- `reduction='mean'`：除以所有元素个数 (包括类别数)，**数学上不正确**
- `reduction='batchmean'`：除以 batch size，**这才是数学上正确的平均 KL 散度**

对于 shape `(3,)` 的单个分布，PyTorch 会把第一维视为 batch size，导致 `batchmean` 结果被除以 3。


In [ ]:
import torch
import torch.nn.functional as F

# 准备数据：3 个类别的概率分布
P = torch.tensor([0.1, 0.4, 0.5])  # 真实分布
logits_Q = torch.tensor([0.2, 0.1, 0.9])  # 预测的 logits

# PyTorch API 用法
log_Q = F.log_softmax(logits_Q, dim=0)  # 必须先转换为 log probabilities

print("=" * 60)
print("PyTorch API 不同 reduction 模式的结果")
print("=" * 60)

kl_sum = F.kl_div(log_Q, P, reduction='sum')
kl_batchmean = F.kl_div(log_Q, P, reduction='batchmean')

print(f"reduction='sum':       {kl_sum.item():.6f}")
print(f"reduction='batchmean': {kl_batchmean.item():.6f}")
print(f"\n注意：batchmean = sum / batch_size = {kl_sum.item():.6f} / 3 = {(kl_sum/3).item():.6f}")

# 手动计算验证
Q = F.softmax(logits_Q, dim=0)
manual_kl = torch.sum(P * (torch.log(P) - torch.log(Q)))
print(f"\n手动公式计算：        {manual_kl.item():.6f}")
print(f"✅ 与 sum 模式一致：{torch.isclose(kl_sum, manual_kl)}")


### 3. 正确的批量数据用法

对于真实的批量数据 (如 shape `(N, C)`)，`batchmean` 才是正确的选择：


In [ ]:
# 批量数据示例：2 个样本，每个 3 个类别
P_batch = torch.tensor([[0.1, 0.4, 0.5], 
                        [0.2, 0.3, 0.5]])
logits_Q_batch = torch.tensor([[0.2, 0.1, 0.9], 
                               [0.5, 0.3, 0.8]])

log_Q_batch = F.log_softmax(logits_Q_batch, dim=1)

kl_batch = F.kl_div(log_Q_batch, P_batch, reduction='batchmean')
print(f"批量 KL 散度 (平均每个样本): {kl_batch.item():.6f}")

# 手动验证：先计算每个样本的 KL，再求平均
Q_batch = F.softmax(logits_Q_batch, dim=1)
kl_sample1 = torch.sum(P_batch[0] * (torch.log(P_batch[0]) - torch.log(Q_batch[0])))
kl_sample2 = torch.sum(P_batch[1] * (torch.log(P_batch[1]) - torch.log(Q_batch[1])))
manual_avg = (kl_sample1 + kl_sample2) / 2

print(f"手动计算 (每个样本的平均): {manual_avg.item():.6f}")
print(f"✅ 结果一致：{torch.isclose(kl_batch, manual_avg)}")


### 4. 手动实现（处理数值稳定性）

完整实现需要处理 $0 \log 0$ 问题和各种 reduction 模式：


In [ ]:
def my_kl_div(input_log_q, target_p, reduction="batchmean"):
    """
    手动实现 KL 散度，等效于 torch.nn.functional.kl_div
    
    Args:
        input_log_q: log(Q)，已经过 log_softmax 处理
        target_p: P，概率分布
        reduction: 'none' | 'sum' | 'mean' | 'batchmean'
    """
    # 处理 P * log(P)，避免 0 * log(0) = NaN
    term_p_log_p = torch.zeros_like(target_p)
    mask = target_p > 0
    term_p_log_p[mask] = target_p[mask] * torch.log(target_p[mask])
    
    # 计算 P * log(Q)
    term_p_log_q = target_p * input_log_q
    
    # KL 散度：P * log(P) - P * log(Q)
    pointwise_kl = term_p_log_p - term_p_log_q
    
    if reduction == "none":
        return pointwise_kl
    elif reduction == "sum":
        return torch.sum(pointwise_kl)
    elif reduction == "mean":
        return torch.mean(pointwise_kl)
    elif reduction == "batchmean":
        batch_size = input_log_q.size(0)
        return torch.sum(pointwise_kl) / batch_size
    else:
        raise ValueError(f"Invalid reduction: {reduction}")

# 验证实现
logits = torch.tensor([[0.5, 0.2, 0.3], [0.1, 0.8, 0.1]])
input_log_q = F.log_softmax(logits, dim=1)
target_p = torch.tensor([[0.1, 0.9, 0.0], [0.2, 0.5, 0.3]])  # 包含 0 测试数值稳定性

loss_official = F.kl_div(input_log_q, target_p, reduction="batchmean")
loss_custom = my_kl_div(input_log_q, target_p, reduction="batchmean")

print(f"官方 API 结果: {loss_official.item():.6f}")
print(f"手动实现结果: {loss_custom.item():.6f}")
print(f"✅ 完全一致：{torch.isclose(loss_official, loss_custom)}")


### 5. 实际应用场景

#### 5.1 变分自编码器 (VAE)
VAE 的损失函数包含 KL 散度正则项，让编码器输出的潜在分布接近标准正态分布 $N(0, 1)$。

#### 5.2 知识蒸馏 (Knowledge Distillation)
让小模型（Student）的输出分布逼近大模型（Teacher）的输出分布，实现知识迁移。

#### 5.3 强化学习 (PPO/TRPO)
限制新旧策略之间的 KL 散度，保证策略更新的稳定性。

#### 5.4 与交叉熵的关系
$$
\text{CrossEntropy}(P, Q) = H(P) + D_{KL}(P || Q)
$$

在监督学习中，真实标签 $P$ 是固定的 one-hot 编码，$H(P) = 0$，因此**最小化交叉熵等价于最小化 KL 散度**。


### 6. 可视化理解

用高斯分布直观展示 KL 散度的含义：


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 1000)

def gaussian(x, mu, sigma):
    return (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# 定义两个分布
mu_p, sigma_p = 0, 1.0
P_dist = gaussian(x, mu_p, sigma_p)

mu_q, sigma_q = 1.5, 1.5
Q_dist = gaussian(x, mu_q, sigma_q)

# 计算 KL 散度贡献
eps = 1e-10
kl_pq_pointwise = P_dist * np.log((P_dist + eps) / (Q_dist + eps))

# 绘图
plt.figure(figsize=(14, 6))

# 子图 1：分布对比
plt.subplot(1, 2, 1)
plt.plot(x, P_dist, label=rf"P (Target): $\mu={mu_p}, \sigma={sigma_p}$", 
         color="blue", linewidth=2)
plt.plot(x, Q_dist, label=rf"Q (Model): $\mu={mu_q}, \sigma={sigma_q}$", 
         color="orange", linewidth=2, linestyle="--")
plt.fill_between(x, P_dist, alpha=0.1, color="blue")
plt.fill_between(x, Q_dist, alpha=0.1, color="orange")
plt.title("Probability Distributions P and Q")
plt.xlabel("x")
plt.ylabel("Probability Density")
plt.legend()
plt.grid(True, alpha=0.3)

# 子图 2：KL 散度贡献
plt.subplot(1, 2, 2)
plt.plot(x, kl_pq_pointwise, label=r"$P(x) \log(P(x)/Q(x))$", color="green", linewidth=2)
plt.fill_between(x, kl_pq_pointwise, 0, where=(kl_pq_pointwise > 0), 
                 color="green", alpha=0.3, label="Positive Contribution")
plt.fill_between(x, kl_pq_pointwise, 0, where=(kl_pq_pointwise < 0), 
                 color="red", alpha=0.3, label="Negative Contribution")

total_kl = np.trapz(kl_pq_pointwise, x) if hasattr(np, "trapezoid") else np.trapz(kl_pq_pointwise, x)
plt.title(f"KL Divergence: $D_{{KL}}(P || Q) \\approx$ {total_kl:.4f}")
plt.xlabel("x")
plt.ylabel("KL Contribution")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 总结

**关键要点：**
1. PyTorch 的 `kl_div` 第一个参数必须是 `log_softmax` 的输出
2. 单个分布用 `reduction='sum'`，批量数据用 `reduction='batchmean'`
3. 必须处理 $0 \log 0$ 问题以避免 NaN
4. KL 散度不对称：$D_{KL}(P || Q) \neq D_{KL}(Q || P)$
5. 交叉熵损失本质上就是在优化 KL 散度
